In [1]:
from narwhals.selectors import categorical
!pip install scikit-learn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,KFold,learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from scipy.stats import entropy
import os

%matplotlib widget
plt.style.use("E:\\BongoDev\\Regression Analysis\\deeplearning.mplstyle")


In [3]:
"""
Machine Learning Algorithm:
1. Supervised learning: data contains labels i.e Regression,Classification
2. Unsupervised Learning: data does not contain lables i.e. Clustering
3. Reinforcement Learning : Machine learns through penalty/reward/feedback systems
    i.e. ChatGPT
    1. Train a Text to Text generation model
    2. Task Specific fine tuning the model using reinforcement learning
"""

ROOT_DIR="E:\\BongoDev\\Regression Analysis"
DATA_DIR=os.path.join(ROOT_DIR,"Data")
Dataset_path=os.path.join(DATA_DIR,'housing.csv')



In [4]:
housing_dataset=pd.read_csv(Dataset_path)
housing_dataset.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [5]:
housing_dataset.columns

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='str')

In [6]:
housing_dataset=housing_dataset[['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus','price']]


In [7]:
housing_dataset.head()

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,price
0,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished,13300000
1,8960,4,4,4,yes,no,no,no,yes,3,no,furnished,12250000
2,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished,12250000
3,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished,12215000
4,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished,11410000


In [8]:
housing_dataset.isnull().sum()

area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
price               0
dtype: int64

In [9]:
housing_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   area              545 non-null    int64
 1   bedrooms          545 non-null    int64
 2   bathrooms         545 non-null    int64
 3   stories           545 non-null    int64
 4   mainroad          545 non-null    str  
 5   guestroom         545 non-null    str  
 6   basement          545 non-null    str  
 7   hotwaterheating   545 non-null    str  
 8   airconditioning   545 non-null    str  
 9   parking           545 non-null    int64
 10  prefarea          545 non-null    str  
 11  furnishingstatus  545 non-null    str  
 12  price             545 non-null    int64
dtypes: int64(6), str(7)
memory usage: 69.2 KB


In [10]:
"""
Normalization:
1. Min-Max Normalization   ( 0 -1)
       xmin=20
       xmax=40
       range=20
       if x=30 then x_norm=(x-xmin)/range  =  (30-20)/20=0.5

2. Standardization/ Z-score  (Mean=0, Std=1,Zero Mean,Unit Variance)  ( DEFAULT)
       mean=20
       std=30
       if =30 then z-score=(x-mean)/std=(30-20)/30=0.33

       for example we have 10 images each 3 X 3
       average image=[_ _ _
                      _ _ _
                      _ _ _
                     ]
       For a single image:
              1. remove average image from that image
              2. normalize by diving the std
"""
# Select columns

numerical_cols=housing_dataset.select_dtypes(include='number').columns.drop('price')
categorical_cols=housing_dataset.select_dtypes(include=['object','str']).columns
numerical_cols

Index(['area', 'bedrooms', 'bathrooms', 'stories', 'parking'], dtype='str')

In [11]:
categorical_cols

Index(['mainroad', 'guestroom', 'basement', 'hotwaterheating',
       'airconditioning', 'prefarea', 'furnishingstatus'],
      dtype='str')

In [12]:
scaler=StandardScaler()

housing_dataset[numerical_cols]=scaler.fit_transform(housing_dataset[numerical_cols])

In [14]:
housing_dataset.head()

,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,price
0,1.046726,1.403419,1.421812,1.378217,yes,no,no,no,yes,1.517692,yes,furnished,13300000
1,1.757010,1.403419,5.405809,2.532024,yes,no,no,no,yes,2.679409,no,furnished,12250000
2,2.218232,0.047278,1.421812,0.224410,yes,no,yes,no,no,1.517692,yes,semi-furnished,12250000
3,1.083624,1.403419,1.421812,0.224410,yes,no,yes,no,yes,2.679409,yes,furnished,12215000
4,1.046726,1.403419,-0.570187,0.224410,yes,yes,yes,no,yes,1.517692,no,furnished,11410000


In [16]:
"""
Categorical cols
1.One-hot encoding
2.Label Encoding
3.Ordinal Encoding ( for data in order)

"""
categorical_cols

Index(['mainroad', 'guestroom', 'basement', 'hotwaterheating',
       'airconditioning', 'prefarea', 'furnishingstatus'],
      dtype='str')

In [21]:
le=LabelEncoder()
for col in categorical_cols:
    housing_dataset[col]=le.fit_transform(housing_dataset[col])

housing_dataset.head()


,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus,price
0,1.046726,1.403419,1.421812,1.378217,1,0,0,0,1,1.517692,1,0,13300000
1,1.757010,1.403419,5.405809,2.532024,1,0,0,0,1,2.679409,0,0,12250000
2,2.218232,0.047278,1.421812,0.224410,1,0,1,0,0,1.517692,1,1,12250000
3,1.083624,1.403419,1.421812,0.224410,1,0,1,0,1,2.679409,1,0,12215000
4,1.046726,1.403419,-0.570187,0.224410,1,1,1,0,1,1.517692,0,0,11410000


In [22]:
housing_dataset['guestroom'].value_counts()

guestroom
0    448
1     97
Name: count, dtype: int64

In [29]:
"""
Population ->sample -->resample(train-test-val)
"""
X=housing_dataset.drop('price',axis=1).values
y=housing_dataset['price'].values
print(X.shape)
print(y.shape)

(545, 12)
(545,)


In [31]:
X_train_val,X_test,y_train_val,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)
X_train_val.shape,y_train_val.shape,X_test.shape,y_test.shape

((436, 12), (436,), (109, 12), (109,))

In [32]:
X_train,X_val,y_train,y_val=train_test_split(
    X_train_val,y_train_val,test_size=0.25,random_state=42
)

In [33]:
print(f"TrainSIZE:{len(X_train)},ValSize:{len(X_val)},TestSize:{len(X_test)}")

TrainSIZE:327,ValSize:109,TestSize:109


In [34]:
"""
Multiple Value Linear Regression Model
y_pred=w*x+b

x=[x1 ,x2 ,x3]
w=[w1,w2,w3]

y_pred=x1*w1+x2*w2+x3*w3+b
"""

def model(x,w,b):
    y_pred=np.dot(x,w)+b
    return y_pred

In [35]:
def cost_function(x,y,w,b):
    y_pred=model(x,w,b)
    mse=np.mean((y-y_pred)**2)
    return mse

In [36]:
def compute_gradients(x,y_true,w,b):
    delta=1e-9
    cost_1=cost_function(x,y_true,w,b)
    cost_2=cost_function(x,y_true,w+delta,b)
    cost_3=cost_function(x,y_true,w,b+delta)
    dw=(cost_2-cost_1)/delta
    db=(cost_3-cost_1)/delta
    return dw,db

In [41]:
# Initialize paramters
np.random.seed(42)
w_init=np.random.randn(X_train.shape[1])*0.01
b_init=0.0

w_init


array([ 0.00496714, -0.00138264,  0.00647689,  0.0152303 , -0.00234153,
       -0.00234137,  0.01579213,  0.00767435, -0.00469474,  0.0054256 ,
       -0.00463418, -0.0046573 ])

In [42]:
# Hyperparameters
learning_rate=0.01
epochs=50000

In [43]:
# Gradient Descent
# In each epoch 1. Loss calculate 2. Gradient Calculate 3.Weight Update